# Phase 3 — Features et baseline

**Objectif** : mesurer la **baseline**, c'est-à-dire le score d'une règle simple que tout modèle
devra battre, et préparer la façon de **transformer un avis en nombres** pour les modèles de la
phase 4. À la fin de ce notebook, on connaît le score à battre, et chaque choix est justifié.

**Avant de commencer**

1. Les jeux propres doivent exister : `uv run python -m review_pilot.data.build` (depuis la racine du projet).
2. Dans VS Code, choisis le noyau (*kernel*) `.venv` en haut à droite du notebook.

**Méthode** : chaque section pose ses questions, puis donne le code, les résultats et une
cellule « Observations ». Les blocs repliés **📖 Comprendre** expliquent les notions utilisées.

> Règle d'or : tout ce qui **apprend** (listes de mots, vocabulaire) apprend sur le **train**
> uniquement. La **validation** sert à mesurer et à régler. Le **test** n'est pas ouvert.

## 0. Le contexte : que fait-on dans cette phase ?

**Où en est le projet.** La phase 1 a fixé le besoin : **signaler au service client les avis
négatifs**, en en trouvant au moins 90 % (rappel) avec au plus ~15 % de fausses alertes
(précision d'au moins 85 %). La phase 2 a produit trois jeux propres : train, validation, test.

**Pourquoi une baseline ?** Un modèle qui trouve 80 % des avis négatifs, est-ce bien ? Impossible
à dire sans point de comparaison. La **baseline** est ce point de comparaison : une méthode
simple, sans apprentissage automatique. Si un modèle compliqué ne la bat pas nettement, il ne
sert à rien. On en mesure deux :

| Baseline | Principe | Rôle |
|---|---|---|
| **Classe majoritaire** | répondre toujours la même chose (la classe la plus fréquente du train) | contrôle minimal : le niveau « zéro effort » |
| **Règle mots-clés** | compter les mots négatifs et positifs de l'avis | la vraie baseline à battre, fixée en phase 1 |

**Les features.** Un modèle ne lit pas : il calcule. Il faut donc transformer chaque avis en une
liste de nombres, les **features**. L'EDA a montré que la forme du texte (longueur, ponctuation)
trahit à peine le sentiment : ce sont les **mots** qui comptent. On prépare ici la représentation
**TF-IDF**, qui servira aux modèles de la phase 4.

**Les données utilisées**

| Jeu | Rôle dans ce notebook |
|---|---|
| train (44 896 avis) | apprendre les listes de mots-clés et le vocabulaire TF-IDF |
| validation (4 989 avis) | mesurer la baseline et choisir ses réglages |
| test (9 975 avis) | **pas ouvert** : il sert une seule fois, en phase 5 |

**Ce que la phase apporte au projet.** Le score à battre (noté dans le suivi du projet et
enregistré dans MLflow), et une transformation du texte prête pour la modélisation.

<details>
<summary>📚 Lexique — les mots techniques de ce notebook</summary>

| Mot | Sens |
|---|---|
| **Baseline** | Une méthode simple qui sert de point de comparaison : un modèle doit faire nettement mieux. |
| **Baseline analytique** | Des taux calculés par groupe d'avis (« part de négatifs chez les avis longs ») pour comprendre ce qui fait varier la cible. |
| **Classe majoritaire** | La classe la plus fréquente du train. Le « modèle » qui la prédit toujours est le plus bête possible. |
| **Segment / tranche** (*bin*) | Un groupe d'avis qui partagent une caractéristique (« entre 30 et 50 mots »). |
| **Quantile** | Les valeurs qui coupent les avis triés en parts égales : 5 tranches de quantiles contiennent chacune 20 % des avis. |
| **Feature** | Une variable numérique qui décrit un avis et que le modèle utilise pour prédire. |
| **Mot-clé** | Ici, un mot beaucoup plus fréquent dans une classe que dans l'autre (« refund » pour les négatifs). |
| **Fréquence documentaire** | Le nombre d'avis qui contiennent un mot (un mot répété dans un avis compte une fois). |
| **Score** | Le nombre que la règle calcule pour un avis : mots négatifs moins mots positifs. |
| **Seuil de décision** | La valeur de score à partir de laquelle on signale l'avis comme négatif. |
| **Signaler** | Prédire « négatif » : l'avis part au service client. |
| **Rappel négatif** (`neg_recall`) | Parmi les avis vraiment négatifs, la part que l'on signale. La métrique principale du projet. |
| **Précision négative** (`neg_precision`) | Parmi les avis signalés, la part qui sont vraiment négatifs. Doit rester ≥ 85 %. |
| **Fausse alerte** (faux positif) | Un avis positif signalé à tort : un conseiller le relit pour rien. |
| **Négatif raté** (faux négatif) | Un avis négatif non signalé : un client mécontent n'est pas traité. Le pire cas. |
| **F1** | Une moyenne du rappel et de la précision d'une classe, qui chute si l'un des deux est mauvais. |
| **Accuracy** | La part de bonnes réponses, toutes classes confondues. |
| **Matrice de confusion** | Le tableau qui croise la vraie classe et la classe prédite : il montre chaque sorte d'erreur. |
| **Réglage** (hyperparamètre) | Un choix fait avant l'apprentissage (nombre de mots-clés, seuil), qu'on règle sur la validation. |
| **Négation** | Un mot qui inverse le sens de la suite : « not », « never », « don't »… |
| **TF-IDF** | Une façon de transformer un avis en nombres : chaque mot reçoit un poids, élevé s'il est fréquent dans l'avis mais rare dans l'ensemble des avis. |
| **N-gramme / bigramme** | Une suite de n mots. Un bigramme est une paire de mots qui se suivent (« not good »). |
| **Vocabulaire** | La liste des mots (et paires de mots) que le TF-IDF connaît : il est appris sur le train. |
| **Matrice creuse** (*sparse*) | Un tableau presque entièrement rempli de zéros, stocké en ne gardant que les valeurs non nulles. |
| **`fit` / `transform`** | `fit` apprend (le vocabulaire, les poids) ; `transform` applique ce qui a été appris à d'autres données. |
| **Fuite de données** | Quand une information de la validation ou du test se glisse dans l'apprentissage : le score devient trop optimiste. |
| **MLflow** | Un outil qui garde le journal de chaque essai de modèle : réglages, scores, fichiers. |
| **Run** (MLflow) | Un essai enregistré : un modèle, ses réglages et ses scores. |
| **Artefact** (MLflow) | Un fichier joint à un run : une figure, une liste de mots… |

</details>

## 1. Imports et chargement

- Quelles librairies faut-il pour compter des mots, mesurer des scores et enregistrer des essais ?
- Les jeux propres sont-ils bien ceux produits en phase 2 (44 896 avis train, 4 989 validation) ?
- Quel texte donne-t-on à lire : le titre, le contenu, ou les deux ?

<details>
<summary>📖 Comprendre</summary>

- `re` et `Counter` servent à découper les avis en mots et à les compter.
- `scikit-learn` fournit les **métriques** (rappel, précision…) et le **TF-IDF**.
- `mlflow` enregistre les essais de modèles (section 7).
- **Titre + contenu** : l'EDA a montré que les titres sont courts mais très chargés de
  sentiment (« Disappointed », « Excellent »). On les colle devant le contenu pour ne rien perdre.
- **Deux constantes du projet** : `NEGATIVE = 0` (la classe à repérer) et `PRECISION_MIN = 0.85`
  (le plancher de précision fixé en phase 1). Les écrire une fois en haut évite de se tromper plus bas.

</details>

In [ ]:
import re
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_colwidth", 120)

# Le notebook s'exécute depuis notebooks/ : le reste du projet est un dossier plus haut.
PROCESSED_DIR = Path("../data/processed")
FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

NEGATIVE = 0  # label 0 = avis négatif : la classe à repérer
PRECISION_MIN = 0.85  # plancher de précision négative fixé en phase 1
SEED = 42

In [ ]:
train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
val = pd.read_parquet(PROCESSED_DIR / "val.parquet")

# Titre + contenu : le titre porte beaucoup de sentiment (EDA, section 4).
for frame in (train, val):
    frame["text"] = frame["title"] + " " + frame["content"]

print("train :", train.shape, "| validation :", val.shape)
print(
    f"avis négatifs : {(train['label'] == NEGATIVE).mean():.2%} (train), "
    f"{(val['label'] == NEGATIVE).mean():.2%} (validation)"
)

**Observations**

- 44 896 avis dans le train et 4 989 dans la validation : ce sont bien les jeux produits en
  phase 2 (`reports/data_quality.md`). La 4ᵉ colonne est `text`, le titre suivi du contenu.
- Même part d'avis négatifs des deux côtés, **50,13 %** : le découpage stratifié a fonctionné.
- Deux messages s'affichent à l'import de MLflow (un avertissement de `tqdm`, la librairie des
  barres de progression, et un conseil sur le « tracing ») : ils sont informatifs, sans effet
  sur les calculs.

## 2. Ce qui fait varier la part d'avis négatifs (baseline analytique)

- La part d'avis négatifs change-t-elle avec la longueur de l'avis ?
- Et avec la présence d'un `?` ou d'un `!` ?
- Ces écarts sont-ils assez grands pour repérer les avis négatifs à eux seuls ?

<details>
<summary>📖 Comprendre</summary>

**L'idée.** Avant toute règle ou tout modèle, on découpe les avis du train en groupes
(**segments**) et on calcule, dans chaque groupe, la **part d'avis négatifs**. Si un groupe en
contient 80 % alors que la moyenne est de 50 %, la caractéristique qui définit ce groupe est
un bon indice.

**Les tranches de longueur.** On trie les avis par nombre de mots et on les coupe en 5 tranches
de même effectif (des **quantiles**) : les 20 % les plus courts, puis les 20 % suivants, etc.
Couper par effectif plutôt que par bornes fixes (0-50, 50-100…) garantit qu'aucune tranche n'est
presque vide.

**Comment lire le tableau.** La colonne « part de négatifs » se compare à la moyenne du train
(50,13 %). Un écart de 2 ou 3 points est faible ; un écart de 20 points serait un indice fort.

**Pourquoi le faire, si l'EDA a déjà calculé des corrélations ?** Une corrélation résume un lien
en un seul nombre, et ne voit bien que les liens en ligne droite. Un taux par tranche montre la
forme du lien : par exemple, si seuls les avis très longs sont plus souvent négatifs.

</details>

In [ ]:
train["words"] = train["content"].str.split().str.len()
train["length_bin"] = pd.qcut(train["words"], q=5)
train["has_question"] = train["text"].str.contains("?", regex=False)
train["has_exclamation"] = train["text"].str.contains("!", regex=False)
train["is_negative"] = train["label"] == NEGATIVE


def segment_label(value: object) -> str:
    """Étiquette lisible : « 6 à 34 » pour une tranche de mots, « oui » / « non » sinon."""
    if isinstance(value, pd.Interval):
        return f"{int(value.left) + 1} à {int(value.right)}"
    return "oui" if value else "non"


SEGMENTS = {
    "longueur (mots)": "length_bin",
    "contient un ?": "has_question",
    "contient un !": "has_exclamation",
}
analytic = pd.concat(
    {
        name: train.groupby(column, observed=True)["is_negative"]
        .agg(["size", "mean"])
        .rename(index=segment_label)
        for name, column in SEGMENTS.items()
    },
    names=["variable", "segment"],
).rename(columns={"size": "avis", "mean": "part de négatifs (%)"})
analytic["part de négatifs (%)"] = (analytic["part de négatifs (%)"] * 100).round(1)
analytic

In [ ]:
fig, axes = plt.subplots(1, len(SEGMENTS), figsize=(14, 4), sharey=True)
overall = train["is_negative"].mean() * 100
for ax, name in zip(axes, SEGMENTS, strict=True):
    rates = analytic.loc[name, "part de négatifs (%)"]
    ax.bar(rates.index, rates.to_numpy(), color="tab:red", alpha=0.7)
    ax.axhline(overall, color="black", linestyle="--", label=f"moyenne ({overall:.1f} %)")
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("part d'avis négatifs (%)")
axes[0].legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_negative_rate_by_segment.png", dpi=120)
plt.show()

**Observations** (moyenne du train : 50,13 % d'avis négatifs)

- **Le `?` est l'indice le plus fort** : 68,3 % des avis qui en contiennent sont négatifs, contre
  48,0 % des autres (+18 points au-dessus de la moyenne). Mais il ne concerne que 4 693 avis,
  soit 10,5 % du train.
- **Le `!` penche vers le positif** : 42,7 % de négatifs parmi les 14 971 avis qui en contiennent
  (33,3 % du train), contre 53,8 % pour les autres.
- **La longueur joue peu** : les 20 % d'avis les plus courts (6 à 34 mots) comptent 43,2 % de
  négatifs ; au-delà de 54 mots, on est entre 52,4 % et 53,8 %. L'écart maximal est de
  10 points, et il plafonne dès 54 mots.
- Ces chiffres confirment l'EDA (section 7) : les avis négatifs posent plus de questions et sont
  un peu plus longs. Mais **aucun segment n'isole les avis négatifs** : le meilleur (le `?`) ne
  touche qu'un avis sur dix, et un tiers de ces avis sont positifs. Ces variables de forme ne
  seront donc pas utilisées comme features (choix fait en début de phase).
- Figure enregistrée : `reports/figures/03_negative_rate_by_segment.png`.

## 3. Les listes de mots-clés, apprises sur le train

- Quels mots sont beaucoup plus fréquents dans les avis négatifs que dans les positifs ?
  Et l'inverse ?
- Les listes obtenues ont-elles du sens quand on les relit ?
- Contiennent-elles des mots qui ne parlent pas de sentiment ?

<details>
<summary>📖 Comprendre</summary>

**Le principe.** Plutôt que de choisir les mots à la main (subjectif), on les **calcule** sur le
train, comme dans l'EDA (section 8) :

1. On découpe chaque avis en mots (minuscules, suites de lettres et apostrophes : « don't »
   reste un seul mot).
2. Pour chaque mot, on compte dans combien d'avis négatifs et dans combien d'avis positifs il
   apparaît : c'est sa **fréquence documentaire**. Un mot répété dix fois dans un même avis ne
   compte qu'une fois, pour qu'un seul client bavard ne fausse pas tout.
3. On calcule un **ratio** : (part des avis négatifs qui contiennent le mot) ÷ (part des avis
   positifs qui le contiennent).

**Exemple** (chiffres du train). « refund » apparaît dans 311 avis négatifs et seulement 6
positifs : ratio de 44, c'est un mot très négatif. « excellent » apparaît dans 313 avis négatifs
et 1 742 positifs : ratio de 0,18, il est environ 5,6 fois plus fréquent chez les positifs.
Un ratio proche de 1 ne dit rien du sentiment : « the » (1,03), « book » (0,87).

**Deux précautions.**
- On ajoute 1 à chaque compte avant de diviser : sinon, un mot absent des avis positifs
  donnerait une division par zéro.
- On ne garde que les mots présents dans **au moins 100 avis** : un mot vu 3 fois, toutes dans
  des avis négatifs, aurait un ratio énorme par pur hasard.

**Pourquoi sur le train seulement ?** Les listes sont apprises. Si on les calculait aussi sur la
validation, la règle aurait « vu » les réponses de la validation, et son score y serait trop beau.

</details>

In [ ]:
TOKEN = re.compile(r"[a-z']+")


def tokenize(text: str) -> list[str]:
    """Découpe en mots, comme dans l'EDA : minuscules, suites de lettres et d'apostrophes."""
    return TOKEN.findall(text.lower())


def document_frequency(tokens: pd.Series) -> Counter[str]:
    """Nombre d'avis qui contiennent chaque mot (un mot répété compte une fois par avis)."""
    return Counter(word for words in tokens for word in set(words))


train_tokens = train["text"].map(tokenize)
val_tokens = val["text"].map(tokenize)

is_negative = train["label"] == NEGATIVE
words = (
    pd.DataFrame(
        {
            "négatifs": pd.Series(document_frequency(train_tokens[is_negative])),
            "positifs": pd.Series(document_frequency(train_tokens[~is_negative])),
        }
    )
    .fillna(0)
    .astype(int)
)
words.index.name = "mot"
words["avis"] = words["négatifs"] + words["positifs"]

MIN_REVIEWS = 100
frequent = words[words["avis"] >= MIN_REVIEWS].copy()
# +1 sur chaque compte : un mot absent d'une classe ne provoque pas de division par zéro.
negative_share = (frequent["négatifs"] + 1) / (is_negative.sum() + 1)
positive_share = (frequent["positifs"] + 1) / ((~is_negative).sum() + 1)
frequent["ratio"] = (negative_share / positive_share).round(2)
# Tri stable par ratio, les égalités départagées par ordre alphabétique : listes reproductibles.
frequent = frequent.sort_index().sort_values("ratio", ascending=False, kind="stable")

print(
    f"{len(words)} mots différents, dont {len(frequent)} présents dans au moins {MIN_REVIEWS} avis"
)

In [ ]:
# Les 25 mots les plus typiques des avis négatifs.
frequent.head(25)

In [ ]:
# Les 25 mots les plus typiques des avis positifs.
frequent.tail(25).iloc[::-1]

**Observations**

- **78 185 mots différents** dans le train, dont seulement **2 453** (3,1 %) apparaissent dans au
  moins 100 avis : ce sont les seuls candidats pour les listes.
- **Mots négatifs** : du vocabulaire de déception (« waste », « worst », « disappointing »,
  « awful ») et du service après-vente (« refund », « defective », « returned », « warranty »,
  « buyer »). C'est cohérent avec l'EDA (section 8).
- **Mots positifs** : de l'enthousiasme (« delightful », « awesome », « excellent », « amazing »,
  « highly »), mais aussi des mots de **thème** plutôt que de sentiment (« resource »,
  « memories », « ages », « brings »), probablement liés aux livres et aux films (non vérifié).
- **Les mots négatifs sont plus tranchés** : le ratio monte jusqu'à 44 côté négatif (« refund »),
  alors que le plus positif (« delightful », 0,14) n'est « que » 7 fois plus fréquent chez les
  positifs. Un client mécontent emploie des mots qu'un client satisfait n'utilise presque jamais.

## 4. La règle mots-clés et son seuil

- Combien de mots-clés garder dans chaque liste ?
- À partir de quel score signale-t-on un avis comme négatif ?
- Quel rappel négatif la règle atteint-elle, en gardant une précision négative d'au moins 85 % ?
- Fait-elle mieux que la classe majoritaire ?

<details>
<summary>📖 Comprendre</summary>

**La règle.** Pour chaque avis : **score = nombre de mots négatifs − nombre de mots positifs**
(un mot répété compte à chaque fois). Si le score atteint le **seuil**, l'avis est signalé.

> Exemple avec un seuil de 2 : « Junk. Stopped working, I want a refund » contient « junk »,
> « stopped » et « refund » (3 mots négatifs) et aucun mot positif : score 3 ≥ 2, l'avis est
> signalé.

**Le seuil règle le compromis entre rappel et précision.**
- Seuil **bas** : on signale beaucoup d'avis. On rate peu de négatifs (bon rappel), mais on
  déclenche beaucoup de fausses alertes (mauvaise précision).
- Seuil **haut** : on ne signale que les avis très clairement négatifs. Peu de fausses alertes,
  mais beaucoup de négatifs ratés.

La phase 1 a fixé la règle du jeu : **le meilleur rappel négatif possible, tant que la précision
négative reste d'au moins 85 %**. On essaie donc chaque seuil sur la validation et on garde le
meilleur qui respecte ce plancher.

**Le nombre de mots-clés** est un autre réglage : peu de mots = une règle très sûre mais qui ne
voit rien dans la plupart des avis ; beaucoup de mots = elle voit tout, mais avec des mots de
moins en moins liés au sentiment. On essaie plusieurs tailles, et pour chacune on cherche son
meilleur seuil.

**Attention, régler sur la validation rend son score un peu optimiste** : on a choisi les
réglages qui y marchent le mieux. C'est pour ça que le test reste fermé : il donnera en
phase 5 le score honnête, pour la baseline comme pour le modèle.

**La classe majoritaire** prédit toujours la classe la plus fréquente du train. Si c'est
« négatif », elle signale **tout** : rappel de 100 %, mais une précision d'environ 50 %.
C'est exactement la solution « tout signaler » que le plancher de précision interdit.

</details>

In [ ]:
def keyword_lists(n_words: int) -> tuple[set[str], set[str]]:
    """Les n_words mots les plus négatifs et les n_words mots les plus positifs."""
    return set(frequent.index[:n_words]), set(frequent.index[-n_words:])


def keyword_score(
    tokens: pd.Series, negative_words: set[str], positive_words: set[str]
) -> pd.Series:
    """Nombre de mots négatifs moins nombre de mots positifs, pour chaque avis."""
    return tokens.map(
        lambda words: (
            sum(w in negative_words for w in words) - sum(w in positive_words for w in words)
        )
    )


def negative_metrics(labels: pd.Series, flagged: pd.Series) -> dict[str, float]:
    """Métriques du projet : l'avis signalé (flagged) est prédit négatif."""
    truly_negative = labels == NEGATIVE
    return {
        "neg_recall": recall_score(truly_negative, flagged),
        "neg_precision": precision_score(truly_negative, flagged, zero_division=0),
        "neg_f1": f1_score(truly_negative, flagged),
        "pos_f1": f1_score(~truly_negative, ~flagged),
        "accuracy": accuracy_score(truly_negative, flagged),
    }


def threshold_table(score: pd.Series, labels: pd.Series) -> pd.DataFrame:
    """Rappel et précision négatifs pour chaque seuil possible du score."""
    rows = []
    for threshold in range(int(score.min()), int(score.max()) + 1):
        flagged = score >= threshold
        rows.append(
            {"seuil": threshold, "signalés": int(flagged.sum())} | negative_metrics(labels, flagged)
        )
    return pd.DataFrame(rows).set_index("seuil")


def best_threshold(table: pd.DataFrame) -> int:
    """Le seuil au meilleur rappel négatif parmi ceux qui gardent la précision au plancher."""
    allowed = table[table["neg_precision"] >= PRECISION_MIN]
    return int(allowed["neg_recall"].idxmax())

In [ ]:
# Pour chaque taille de liste : le meilleur seuil, et les scores obtenus sur la validation.
N_WORDS_GRID = [25, 50, 100, 200, 400, 800]
grid_rows = []
for n_words in N_WORDS_GRID:
    negative_words, positive_words = keyword_lists(n_words)
    table = threshold_table(keyword_score(val_tokens, negative_words, positive_words), val["label"])
    threshold = best_threshold(table)
    grid_rows.append(
        {"mots par liste": n_words, "seuil": threshold} | table.loc[threshold].to_dict()
    )
grid = pd.DataFrame(grid_rows).set_index("mots par liste").astype({"signalés": int})
grid.round(3)

In [ ]:
# La taille retenue : le meilleur rappel négatif de la grille (la précision est déjà au plancher).
N_WORDS = int(grid["neg_recall"].idxmax())
negative_words, positive_words = keyword_lists(N_WORDS)
val_score = keyword_score(val_tokens, negative_words, positive_words)
val_table = threshold_table(val_score, val["label"])
THRESHOLD = best_threshold(val_table)
print(f"mots par liste : {N_WORDS} | seuil : {THRESHOLD}")

# Les seuils autour du seuil retenu : le compromis rappel / précision.
val_table.loc[THRESHOLD - 3 : THRESHOLD + 3].round(3)

In [ ]:
# Les 20 derniers mots retenus dans chaque liste (les moins marqués) : parlent-ils de sentiment ?
ranks = f"rangs {N_WORDS - 19} à {N_WORDS}"
pd.DataFrame(
    {
        f"négatifs ({ranks})": frequent.index[N_WORDS - 20 : N_WORDS],
        f"positifs ({ranks})": frequent.index[-N_WORDS:][::-1][N_WORDS - 20 :],
    }
)

In [ ]:
majority_label = int(train["label"].mode()[0])
majority_flagged = pd.Series(majority_label == NEGATIVE, index=val.index)
keyword_flagged = val_score >= THRESHOLD

results = pd.DataFrame(
    {
        f"classe majoritaire ({majority_label})": negative_metrics(val["label"], majority_flagged),
        "règle mots-clés": negative_metrics(val["label"], keyword_flagged),
    }
).T
results.round(3)

In [ ]:
predicted_label = np.where(keyword_flagged, NEGATIVE, 1)
confusion_figure, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    val["label"],
    predicted_label,
    display_labels=["négatif (0)", "positif (1)"],
    cmap="Blues",
    ax=ax,
)
ax.grid(False)  # le quadrillage du thème seaborn se superposerait aux cases
ax.set_xlabel("classe prédite par la règle")
ax.set_ylabel("vraie classe")
ax.set_title("Règle mots-clés sur la validation")
confusion_figure.tight_layout()
confusion_figure.savefig(FIGURES_DIR / "03_confusion_matrix_keywords.png", dpi=120)
plt.show()

**Observations**

- **Taille des listes** : le rappel négatif monte avec le nombre de mots jusqu'à 400 par liste
  (77,0 %), puis chute à 800 (60,1 %). À 800, les mots ajoutés sont presque neutres : il faut un
  seuil de 7 pour garder la précision au plancher, et on rate alors beaucoup de négatifs.
- **Réglage retenu : 400 mots par liste, seuil 2.** Sur la validation : **rappel négatif 77,0 %**,
  **précision négative 87,6 %**, accuracy 83,0 %.
- **Le compromis du seuil** : le seuil 1 donnerait 84,8 % de rappel, mais 83,3 % de précision,
  sous le plancher de 85 % : il est refusé. Passer de 1 à 2 coûte 7,8 points de rappel pour
  4,3 points de précision ; plus haut, chaque cran coûte encore plus de rappel (10 à 11 points)
  pour moins de précision gagnée (2 à 3 points).
- **Les derniers mots retenus parlent peu de sentiment** : « box », « pad », « connect »,
  « upgrade » côté négatif ; « history », « politics », « king », « chair » côté positif. La règle
  apprend donc en partie le **thème** de l'avis (du matériel est plus souvent critiqué, un livre
  d'histoire plus souvent apprécié : interprétation non vérifiée). Autres surprises : « not » est
  un mot-clé négatif, « price » et « recommend » des mots-clés positifs.
- **Classe majoritaire** : le train compte 50,13 % d'avis négatifs, la classe majoritaire est donc
  « négatif ». Elle signale tout : rappel 100 %, mais précision 50,1 % et F1 positif nul. Elle
  est exclue par le plancher de précision, comme prévu.
- **Matrice de confusion** : sur 2 501 avis négatifs, 1 926 sont signalés et **575 ratés** ; sur
  2 488 positifs, 272 fausses alertes.
- **Distance à l'objectif** : il faut 90 % de rappel, soit environ 2 251 négatifs trouvés ; la
  règle en trouve 1 926. **Il manque 13 points, environ 325 avis négatifs.**
- Figure enregistrée : `reports/figures/03_confusion_matrix_keywords.png`.

## 5. Où la règle se trompe

- Combien d'avis ne contiennent **aucun** mot-clé ? Que devient-on avec eux ?
- Les négatifs ratés contiennent-ils plus souvent une négation devant un mot positif
  (« not recommend ») que les autres avis négatifs ?
- À quoi ressemblent concrètement un négatif raté et une fausse alerte ?

<details>
<summary>📖 Comprendre</summary>

**Pourquoi étudier les erreurs d'une simple baseline ?** Parce que ses faiblesses disent ce
qu'un vrai modèle devra savoir faire mieux. Si la règle rate surtout les avis sans mot-clé, il
faut un modèle qui exploite **tous** les mots ; si elle se fait piéger par les négations, il faut
un modèle qui regarde **les mots qui se suivent**.

**Les deux sortes d'erreurs**, dans le vocabulaire du projet :
- **négatif raté** : un avis négatif dont le score reste sous le seuil. Le client mécontent n'est
  pas traité : c'est l'erreur la plus coûteuse ;
- **fausse alerte** : un avis positif signalé. Un conseiller le relit pour rien.

**Le piège de la négation.** La règle compte les mots un par un, sans regarder ce qui les
précède. Dans « I would not recommend it », « not » compte comme négatif et « recommend » comme
positif : ils s'annulent, score 0, alors que l'avis est clairement négatif. On compte les avis où une négation (« not », « no », « never »,
ou un mot en « n't ») précède un mot-clé positif à trois mots ou moins.

**Comparer avant de conclure** : si 30 % des négatifs ratés contiennent ce piège, c'est
beaucoup… seulement si c'est nettement plus que chez les négatifs bien signalés.

</details>

In [ ]:
keywords = negative_words | positive_words
val_hits = val_tokens.map(lambda words: sum(w in keywords for w in words))
no_keyword = val_hits == 0
val_negative = val["label"] == NEGATIVE
missed = val_negative & ~keyword_flagged
false_alarm = ~val_negative & keyword_flagged

print(f"avis sans aucun mot-clé : {no_keyword.sum()} ({no_keyword.mean():.1%} de la validation)")
print(f"  dont négatifs : {(no_keyword & val_negative).sum()}")
print(f"négatifs ratés : {missed.sum()} sur {val_negative.sum()} négatifs")
print(f"  dont sans aucun mot-clé : {(missed & no_keyword).sum()}")
print(f"fausses alertes : {false_alarm.sum()}")

In [ ]:
NEGATIONS = {"not", "no", "never"}
WINDOW = 3


def negated_positive(words: list[str]) -> bool:
    """Une négation précède-t-elle un mot-clé positif à WINDOW mots ou moins ?"""
    for i, word in enumerate(words):
        if word in positive_words:
            before = words[max(i - WINDOW, 0) : i]
            if any(w in NEGATIONS or w.endswith("n't") for w in before):
                return True
    return False


has_negated_positive = val_tokens.map(negated_positive)
caught = val_negative & keyword_flagged
pd.DataFrame(
    {
        "avis": [missed.sum(), caught.sum()],
        "avec négation + mot positif": [
            (missed & has_negated_positive).sum(),
            (caught & has_negated_positive).sum(),
        ],
    },
    index=["négatifs ratés", "négatifs signalés"],
).assign(part=lambda t: (t["avec négation + mot positif"] / t["avis"]).round(3))

In [ ]:
def show(mask: pd.Series, n: int = 3) -> None:
    """Affiche n avis tirés au hasard, avec leur score et les mots-clés trouvés."""
    for index in val[mask].sample(n, random_state=SEED).index:
        found = [w for w in val_tokens[index] if w in keywords]
        print(f"[{index}] score {val_score[index]} | mots-clés : {found}")
        print("   ", val.loc[index, "text"][:300], "\n")


print("=== Négatifs ratés ===")
show(missed)
print("=== Fausses alertes ===")
show(false_alarm)

**Observations**

- **Presque tous les avis contiennent au moins un mot-clé** : seuls 18 (0,4 %) n'en ont aucun,
  dont 8 négatifs. Avec 800 mots-clés, la règle « voit » quelque chose dans chaque avis : ses
  erreurs ne viennent pas d'un manque de mots, mais de **mots mal pesés**. Sur les 575 négatifs
  ratés, 8 seulement n'ont aucun mot-clé.
- **Négation devant un mot positif** : 20,0 % des négatifs ratés (115 sur 575), contre 14,6 % des
  négatifs signalés (282 sur 1 926). Le piège est plus fréquent chez les ratés, mais l'écart est
  modéré (5,4 points) : il explique une partie des ratés, pas la majorité.
- **Ce que montrent les exemples** (3 de chaque sorte, tirés au hasard : à ne pas généraliser) :
  - [1453] « Laughably bad » : « bad » et « star » (négatifs) pèsent moins que « laugh »,
    « album » (deux fois) et « humor » (positifs). Tous les mots comptent pareil, même ceux qui
    ne disent rien du sentiment.
  - [2331] « one of the stupidest horror movies » : « stupidest » est trop rare pour être un
    mot-clé, et « excellent » (dans « had excellent ideas », au passé) annule « not ».
  - [2169] une fausse alerte sur un moulin à café, apprécié par son auteur : les mots du produit
    (« counter », « lid », « motor », « model ») et « not » (deux fois) sont comptés comme
    négatifs.
  - [1515] une fausse alerte qui insulte un autre commentateur, avec le label positif : c'est
    plutôt un avis **hors sujet** que le signe d'une erreur de la règle.
- **Pour la phase 4** : il faut un modèle qui donne **un poids différent à chaque mot** (et un
  poids presque nul aux mots de thème), et qui voit les **mots qui se suivent** (« not only »,
  « not recommend »).

## 6. Transformer un avis en nombres : le TF-IDF

- En combien de nombres un avis est-il transformé ? Combien sont non nuls ?
- Les paires de mots comme « not good » sont-elles bien présentes ?
- Quels mots pèsent le plus lourd dans un avis donné ? Parlent-ils de sentiment ?
- Que se passerait-il si on apprenait le vocabulaire sur le train **et** la validation ?

<details>
<summary>📖 Comprendre</summary>

**Pourquoi autre chose que les mots-clés ?** La règle n'utilise que quelques centaines de mots
et les compte tous pareil. Un modèle de la phase 4 pourra utiliser **tous** les mots et
**apprendre lui-même** le poids de chacun. Il lui faut une représentation numérique de l'avis :
le **TF-IDF**.

**Le TF-IDF, pas à pas.**
1. On établit le **vocabulaire** : tous les mots (et paires de mots) vus dans au moins 5 avis
   du train. Chacun devient une colonne.
2. Pour un avis, chaque colonne reçoit un poids :
   - **TF** (*term frequency*) : le mot est-il présent, et combien de fois, dans l'avis ?
   - **IDF** (*inverse document frequency*) : le mot est-il **rare** dans l'ensemble des avis ?
     « the » est partout : poids presque nul. « refund » est rare : poids élevé.
3. Les poids d'un avis sont ramenés à une même échelle, pour qu'un avis long ne pèse pas plus
   qu'un avis court.

**Les paires de mots** (`ngram_range=(1, 2)`) : en plus des mots seuls, on garde les paires de
mots qui se suivent. « not good » devient une colonne à part, distincte de « good » : c'est ce
qui permettra au modèle de comprendre la négation que la règle mots-clés rate.

**Les réglages choisis** (à affiner en phase 4) :

| Réglage | Valeur | Pourquoi |
|---|---|---|
| `ngram_range` | `(1, 2)` | mots seuls + paires, pour les négations |
| `min_df` | `5` | un mot vu dans moins de 5 avis n'apprend rien (la moitié du vocabulaire n'apparaît qu'une fois, EDA section 8) |
| `sublinear_tf` | `True` | un mot répété 10 fois ne pèse pas 10 fois plus, mais un peu plus (logarithme) |

**Un piège : le découpage de scikit-learn.** Par défaut, il ne garde que les mots d'au moins
deux lettres ou chiffres, sans apostrophe : « don't » devient « don » (le « t » est trop court).
Ce n'est pas grave, « don » garde la trace de la négation, mais il faut le savoir.

**`fit` sur le train, `transform` sur le reste.** Le vocabulaire et les poids IDF sont
**appris** : on les apprend sur le train (`fit_transform`), puis on les **applique** tels quels
à la validation (`transform`). Un mot qui n'existe que dans la validation est ignoré : c'est
normal, un vrai modèle ne connaît pas à l'avance les avis qu'il recevra.

</details>

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)
X_train = vectorizer.fit_transform(train["text"])  # apprend le vocabulaire sur le train
X_val = vectorizer.transform(val["text"])  # applique le même vocabulaire, sans rien apprendre

vocabulary = vectorizer.get_feature_names_out()
bigrams = sum(" " in term for term in vocabulary)
print(f"train : {X_train.shape[0]} avis, {X_train.shape[1]} colonnes")
print(f"validation : {X_val.shape[0]} avis, {X_val.shape[1]} colonnes (même vocabulaire)")
print(f"vocabulaire : {len(vocabulary) - bigrams} mots seuls + {bigrams} paires de mots")
print(f"valeurs non nulles par avis, en moyenne : {X_train.nnz / X_train.shape[0]:.0f}")
print(f"part de valeurs non nulles : {X_train.nnz / np.prod(X_train.shape):.4%}")

In [ ]:
# Les négations sont-elles présentes sous forme de paires ?
for term in ["not good", "not recommend", "never again", "don't", "don", "waste of"]:
    print(f"{term!r:18} dans le vocabulaire : {term in vectorizer.vocabulary_}")

In [ ]:
# Les 10 termes qui pèsent le plus dans un avis de la validation.
example = 0
weights = X_val[example].toarray()[0]
top = weights.argsort()[::-1][:10]
print(val.loc[example, "text"][:300], "\n")
pd.DataFrame({"terme": vocabulary[top], "poids": weights[top].round(3)})

In [ ]:
# Démonstration de fuite : on apprend (à tort) le vocabulaire sur le train ET la validation.
leaky = TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)
leaky.fit(pd.concat([train["text"], val["text"]]))
only_with_val = sorted(set(leaky.vocabulary_) - set(vectorizer.vocabulary_))
print(f"termes qui n'existent que parce que la validation a été vue : {len(only_with_val)}")
print(only_with_val[:15])

**Observations**

- **Chaque avis devient une ligne de 94 943 nombres** : 18 707 mots seuls et 76 236 paires de mots.
  Un avis n'en utilise en moyenne que **98** : 0,10 % des valeurs sont non nulles. Stocké en entier,
  le tableau du train aurait 4,26 milliards de cases (environ 34 Go) ; la **matrice creuse** ne
  garde que les valeurs non nulles.
- **Les négations sont bien là** : « not good », « not recommend », « never again » et « waste of »
  sont des colonnes du vocabulaire. « don't » n'y est pas : il devient « don », comme prévu.
- **Le TF-IDF mesure la rareté, pas le sentiment** : dans l'avis d'exemple (une imprimante), les
  10 termes les plus lourds sont des mots du produit (« color ink », « pixma », « ink cartridge »)
  ou des paires sans intérêt (« though which »). C'est le modèle de la phase 4 qui apprendra quels
  termes comptent pour le sentiment.
- **La fuite, en chiffres** : apprendre le vocabulaire sur le train **et** la validation ajoute
  **8 526 termes** (9 % du vocabulaire) qui n'existent que grâce à la validation, et modifie les
  poids IDF de tous les autres. Le modèle serait évalué sur des avis qu'il a en partie « vus ».
  Dans le code du projet, un test vérifiera que le vocabulaire vient du train seul.

## 7. Enregistrer les baselines dans MLflow

- Comment garder une trace de chaque essai (réglages et scores) pour les comparer plus tard ?
- Où MLflow range-t-il ces informations ?
- Comment retrouver les essais, dans le code et dans l'interface ?

<details>
<summary>📖 Comprendre</summary>

**Le problème.** En phase 4, on essaiera des dizaines de modèles et de réglages. Noter les
scores à la main dans un tableau, c'est l'assurance d'en perdre ou de se tromper. **MLflow**
tient ce journal automatiquement.

**Le vocabulaire de MLflow.**

| Mot | Sens | Ici |
|---|---|---|
| **Expérience** | un dossier qui regroupe des essais comparables | `phase-3-baseline` |
| **Run** | un essai | « classe majoritaire », « règle mots-clés » |
| **Paramètres** | les réglages de l'essai | nombre de mots, seuil |
| **Métriques** | les scores obtenus | `neg_recall`, `neg_precision`… |
| **Artefacts** | des fichiers joints | la matrice de confusion, les listes de mots |

**Où c'est stocké.** Dans le projet, jamais versionné (voir `.gitignore`) :
- `mlflow.db` : une petite base de données (SQLite) qui contient les runs, paramètres et métriques ;
- `mlartifacts/` : les fichiers joints.

**Voir les runs dans une interface web.** Depuis la racine du projet :
`uv run mlflow ui --backend-store-uri sqlite:///mlflow.db`, puis ouvrir
http://127.0.0.1:5000 dans le navigateur.

**Chaque « Run All » ajoute de nouveaux runs**, datés : c'est normal, MLflow garde tout.
L'avertissement de SQLAlchemy (une librairie utilisée par MLflow) est masqué : il concerne le
code interne de MLflow, pas le nôtre.

</details>

In [ ]:
# Avertissement interne à MLflow (option dépréciée de SQLAlchemy 2.1) : sans effet pour nous.
warnings.filterwarnings(
    "ignore", message="The ``noload`` loader strategy", category=DeprecationWarning
)

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
EXPERIMENT = "phase-3-baseline"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    # Fichiers joints à la racine du projet, quel que soit le dossier d'où l'on lance.
    mlflow.create_experiment(
        EXPERIMENT, artifact_location=Path("../mlartifacts").resolve().as_uri()
    )
experiment = mlflow.set_experiment(EXPERIMENT)
print(f"expérience : {experiment.name} | fichiers joints : {experiment.artifact_location}")

In [ ]:
SOURCE = "notebooks/03_features_baseline.ipynb"

with mlflow.start_run(run_name="majority"):
    mlflow.set_tag("source", SOURCE)
    mlflow.log_params({"model": "majority", "majority_label": majority_label})
    mlflow.log_metrics(negative_metrics(val["label"], majority_flagged))

with mlflow.start_run(run_name="keywords"):
    mlflow.set_tag("source", SOURCE)
    mlflow.log_params(
        {
            "model": "keywords",
            "n_words": N_WORDS,
            "min_reviews": MIN_REVIEWS,
            "threshold": THRESHOLD,
        }
    )
    mlflow.log_metrics(negative_metrics(val["label"], keyword_flagged))
    mlflow.log_figure(confusion_figure, "confusion_matrix.png")
    mlflow.log_dict(
        {"negative": sorted(negative_words), "positive": sorted(positive_words)}, "keywords.json"
    )

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=["start_time DESC"])
runs[
    [
        "tags.mlflow.runName",
        "params.n_words",
        "params.threshold",
        "metrics.neg_recall",
        "metrics.neg_precision",
        "metrics.accuracy",
    ]
].head(2).round(3)

**Observations**

- L'expérience `phase-3-baseline` est créée dans `mlflow.db`, avec ses fichiers joints dans
  `mlartifacts/` à la racine du projet.
- Deux runs sont enregistrés et relus : **keywords** (400 mots, seuil 2, rappel 0,770, précision
  0,876, accuracy 0,830) et **majority** (rappel 1,00, précision 0,501). Ce sont exactement les
  scores de la section 4.
- Le run `keywords` contient aussi la matrice de confusion et les deux listes de mots
  (`keywords.json`) : on pourra toujours savoir quels mots ont fait signaler un avis.

## 8. Conclusions

- Quel est le score à battre, et à quelle distance de l'objectif du projet est-il ?
- Qu'a-t-on appris des erreurs de la règle pour la phase 4 ?
- Que faut-il porter dans le code du projet (`src/`) ?

<details>
<summary>📖 Comprendre</summary>

- Ce notebook sert à **décider et mesurer**. Une fois validé, ce qui est retenu est recopié dans
  `src/review_pilot/`, testé, et une seule commande recalcule la baseline et l'enregistre dans
  MLflow. Le notebook reste tel quel : c'est la trace du raisonnement.
- Le code du projet devra retrouver **exactement** les mêmes scores sur la validation.

</details>

### Le score à battre (validation)

| Baseline | Rappel négatif | Précision négative | Accuracy | Verdict |
|---|---|---|---|---|
| Classe majoritaire (« négatif ») | 100 % | 50,1 % | 50,1 % | exclue : sous le plancher de précision |
| **Règle mots-clés** (400 mots par liste, seuil 2) | **77,0 %** | **87,6 %** | 83,0 % | **la baseline à battre** |
| Objectif du projet (phase 1) | ≥ 90 % | ≥ 85 % | | il manque 13 points de rappel |

Ce score est **un peu optimiste** : la taille des listes et le seuil ont été choisis sur la
validation. Le test (phase 5) donnera le score honnête, pour la baseline comme pour le modèle.

### Constats et actions

| Thème | Constat | Section | Action |
|---|---|---|---|
| Forme du texte | le `?` et la longueur ne font varier la part de négatifs que de 10 à 18 points | 2 | pas de variables de forme dans les features |
| Mots-clés | les meilleurs mots sont du vocabulaire de déception et de SAV | 3 | ils servent de baseline explicable |
| Mots-clés | à 400 mots par liste, les derniers mots retenus décrivent le **thème** plus que le sentiment | 4 | le modèle devra pondérer chaque mot |
| Erreurs | 575 négatifs ratés, presque tous avec des mots-clés : les mots sont mal pesés | 5 | un poids appris par terme (phase 4) |
| Erreurs | négation devant un mot positif : 20,0 % des ratés contre 14,6 % des signalés | 5 | paires de mots dans le TF-IDF |
| TF-IDF | 94 943 termes dont 76 236 paires ; « not good » et « not recommend » présents | 6 | représentation retenue pour la phase 4 |
| Fuite | apprendre le vocabulaire avec la validation ajoute 8 526 termes | 6 | `fit` sur le train seul, vérifié par un test |

### Pistes pour la phase 4

- Un premier modèle **linéaire** sur le TF-IDF (régression logistique) : il apprend un poids par
  terme, ce qui le rend **explicable** (contrainte de la phase 1), et il s'entraîne en quelques
  secondes.
- **Validation croisée en 5 plis** sur le train pour comparer les modèles et leurs réglages
  (TF-IDF compris), puis seuil de décision réglé sur la validation, comme ici.
- Surveiller les **mots de thème** dans les poids du modèle.

### Questions encore ouvertes

- La règle, comme le futur modèle, s'appuie en partie sur le thème du produit : serait-elle aussi
  bonne sur des catégories de produits absentes du dataset ? Non vérifiable ici : les données
  n'ont pas de catégorie de produit.
- Les avis hors sujet ou mal notés (comme [1515]) limitent le score maximal atteignable : leur
  part reste inconnue. À estimer lors de l'analyse d'erreurs (phase 6).
- Les réglages du TF-IDF (`min_df`, paires de mots) sont des choix raisonnables, pas des choix
  optimisés : ils seront comparés en phase 4.

### Ce qui sera porté dans le code du projet (`src/review_pilot/`)

| Fichier | Contenu (repris de ce notebook) |
|---|---|
| `features/text.py` | le texte lu (titre + contenu) et le TF-IDF avec les réglages de la section 6 |
| `models/keywords.py` | la règle mots-clés sous la forme d'un modèle scikit-learn : `fit` calcule les listes sur le train, `predict` applique le score et le seuil |
| `evaluation/metrics.py` | les métriques du projet (`neg_recall`, `neg_precision`…) et le choix du seuil sous le plancher de précision |
| `models/baseline.py` | la commande `uv run python -m review_pilot.models.baseline` : mesure les deux baselines sur la validation et les enregistre dans MLflow |

Le code du projet devra retrouver **exactement** les mêmes chiffres : 400 mots par liste, seuil 2,
rappel négatif 0,770, précision négative 0,876. Ce notebook reste tel quel après le portage : son
code est volontairement dupliqué dans `src/`, où il sera testé.